# Sprint 5

## Install PySpark

In [1]:
import sys
import subprocess
import importlib.util
from pathlib import Path

if importlib.util.find_spec("pyspark") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
    print("Installed pyspark.")
else:
    print("pyspark is already available.")

pyspark is already available.


## Spark session

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CS131_bladdards")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Spark version: 4.1.1
Shuffle partitions: 8


## Import the funtions and create data path

In [3]:
from pathlib import Path

import pyspark.sql.functions as F 
from pyspark.sql.functions import (
    col,
    to_date,
    year,
    avg,
    count,
    broadcast,
    substring,
    min as spark_min,
    max as spark_max,
    mean,
    approx_percentile
)

#Added directions for testing
DATA_DIR = Path("../data/MIMIC-IV/hosp")
GENERAL_DATA_DIR = Path("../data")
EVIDENCE_DIR = Path("../out/evidence")

### Prompting file locations

In [4]:
# --- Prompt for data directory locations ---
def prompt_dir(prompt_text):
    val = input(f"{prompt_text}: ").strip()
    return Path(val).resolve() # resolve() turns ../data/MIMIC-IV/hosp into the full absolute path automatically


# Prompt user for each data location
DATA_DIR = prompt_dir("Path to MIMIC hosp dataset")
GENERAL_DATA_DIR = prompt_dir("Path to general data directory")
EVIDENCE_DIR = prompt_dir("Path to out/evidence directory")

# Required files from the MIMIC hospital dataset
hosp_files = ["d_icd_diagnoses.csv.gz", "admissions.csv.gz", "diagnoses_icd.csv.gz", "patients.csv.gz"]
# Required files from the general data directory
general_files = ["bc_icd_codes.csv", "symptom_icd_list.txt"]
# Required files from out/evidence
evidence_files = ["pre_bc_symptom_timeline.tsv"]

print("\nChecking MIMIC hosp files in:", DATA_DIR)
for f in hosp_files:
    p = DATA_DIR / f
    print(f"  {'OK' if p.exists() else 'ERROR: missing'}  {f}")

print("\nChecking general data files in:", GENERAL_DATA_DIR)
for f in general_files:
    p = GENERAL_DATA_DIR / f
    print(f"  {'OK' if p.exists() else 'ERROR: missing'}  {f}")

print("\nChecking out/evidence files in:", EVIDENCE_DIR)
for f in evidence_files:
    p = EVIDENCE_DIR / f
    print(f"  {'OK' if p.exists() else 'ERROR: missing'}  {f}")



Checking MIMIC hosp files in: C:\CS131\bladdards-MIMIC-health\data\MIMIC-IV\hosp
  OK  d_icd_diagnoses.csv.gz
  OK  admissions.csv.gz
  OK  diagnoses_icd.csv.gz
  OK  patients.csv.gz

Checking general data files in: C:\CS131\bladdards-MIMIC-health\data
  OK  bc_icd_codes.csv
  OK  symptom_icd_list.txt

Checking out/evidence files in: C:\CS131\bladdards-MIMIC-health\out\evidence
  OK  pre_bc_symptom_timeline.tsv


## Dataframes

In [4]:
# -------------------------------------------------------
# Visits dataframe (Ara)
# -------------------------------------------------------

# REDEFINING F
# soH (Source of Help): https://spark.apache.org/docs/latest/api/python/user_guide/dataprep.html


# 1) READING THE HOSPITAL SOURCE FILES

# deriving (hospital-admission info) -> subject_id, adm_id, admittime, and race from admissions.csv.gz
admissions = spark.read.csv(
    str(DATA_DIR / "admissions.csv.gz"),
    header=True,
    inferSchema=True)

# deriving (patient-level info) -> subject_id, gender, anchor_age, anchor_year from patients.csv.gz
patients = spark.read.csv(
    str(DATA_DIR / "patients.csv.gz"),
    header=True,
    inferSchema=True)

# reading sprint3 output timeline
pre_bc_symptom_timeline = spark.read.csv(
    "../out/evidence/pre_bc_symptom_timeline.csv",
    header=True,
    inferSchema=True
)

# Extracting relevant visit-level rows from the timeline.
# The below takes the timeline dataset, and only retrieves subject_id, hadm_id, and row_type (keep in mind, I renamed row_tyoe to visit_type to match logic)
# It creates a list of visits of interest that can be joined into hospital data.
visits_labeled = (
    pre_bc_symptom_timeline
    .select("subject_id", "hadm_id", "row_type")
    .withColumnRenamed("row_type", "visit_type")
)


# Here, each row represents one patient admission of interest, along with demographics and derived timing (adge + admit_day) based on the previous timeline
# The final dataframe contains one row per (subject_id, hadm_id, visit_type); and duplicate visit entries are dropped for the same patient, admission, and visit type (as to not inflate the cleaned dataframe with duplicates)
visits = (
    visits_labeled
    .join(
        admissions.select("subject_id", "hadm_id", "admittime", "race"),
        on=["subject_id", "hadm_id"],
        how="inner"
    )
    .join(
        patients.select(
            "subject_id",
            "gender",
            "anchor_age",
            "anchor_year",
            "anchor_year_group"
        ),
        on="subject_id",
        how="inner"
    )
    .withColumn("admit_date", to_date("admittime"))
    .withColumn("approx_year", substring("anchor_year_group", 1, 4))
    .withColumn(
        "admit_day", # refined due to Sharon's commment; although, I am a little worried about implementation here becuase `anchor_year` and `approx_age` are not full dates, but just years. Not sure if Spark will make a fuss about the mismatch
        col("admit_date") - to_date("anchor_year") + to_date("approx_year") #has a wobble of +/- a day which is acceptable for these purposes as it will be consistent
    )
    .withColumn(
        "age",
        (
            col("anchor_age") + ( year("admit_date") - col("anchor_year"))
        ).cast("int")
    )
    .select(
        F.col("subject_id").cast("int").alias("subject_id"),
        F.col("hadm_id").cast("int").alias("hadm_id"),
        F.col("race").cast("string").alias("race"),
        F.col("gender").cast("string").alias("gender"),
        F.col("visit_type").cast("string").alias("visit_type"),
        F.col("age").cast("int").alias("age"),
        F.col("admit_day")
    )
    .dropDuplicates(["subject_id", "hadm_id", "visit_type"])
)

visits.printSchema()
visits.show(20, truncate=False)

output_dir = Path("../data/visits")

if output_dir.exists():
    print ("Parquet file exists for visits")
else:
    visits.write.mode("overwrite").parquet(str(output_dir))
    print(f"Wrote results to: {output_dir}")


root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- visit_type: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- admit_day: date (nullable = true)

+----------+--------+----------------------------------+------+-----------+---+----------+
|subject_id|hadm_id |race                              |gender|visit_type |age|admit_day |
+----------+--------+----------------------------------+------+-----------+---+----------+
|10001401  |21544441|WHITE                             |F     |BC_FIRST_DX|89 |2014-06-04|
|10015568  |26581506|BLACK/AFRICAN                     |M     |BC_FIRST_DX|65 |2011-08-19|
|10024451  |22358047|WHITE                             |M     |BC_FIRST_DX|70 |2020-09-14|
|10024483  |27517184|BLACK/AFRICAN AMERICAN            |M     |BC_FIRST_DX|82 |2008-07-03|
|10026950  |28254249|WHITE                             |M     |BC_FIRST_DX|91 |2011

In [5]:
# -------------------------------------------------------
# BHOOMIKA'S SECTION — diagnoses DataFrame
# Collects all diagnoses for visits of interest
# (pre-BC symptom visits + first BC diagnosis visits)
# -------------------------------------------------------

# STEP 1: Load visits of interest from pre_bc_symptom_timeline
# row_type (SYMPTOM or BC_FIRST_DX) becomes visit_type
timeline_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(EVIDENCE_DIR / "pre_bc_symptom_timeline.csv"))
    .select(
        col("subject_id").cast("int"),
        col("hadm_id").cast("int"),
        col("row_type").alias("visit_type")
    )
    .dropDuplicates(["hadm_id"])  # one visit_type label per admission
)

print("Timeline visits of interest:", timeline_df.count())
timeline_df.show(5)

# STEP 2: Load all diagnoses from MIMIC
# seq_num = order diagnoses were recorded per visit → becomes ranking
dx_icd_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(DATA_DIR / "diagnoses_icd.csv.gz"))
    .select(
        col("subject_id").cast("int"),
        col("hadm_id").cast("int"),
        col("seq_num").cast("int").alias("ranking"),
        col("icd_code"),
        col("icd_version").cast("int")
    )
)

print("diagnoses_icd rows:", dx_icd_df.count())
dx_icd_df.show(5)

# STEP 3: Load ICD code dictionary
# Maps icd_code + icd_version → human readable description
icd_dict_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(DATA_DIR / "d_icd_diagnoses.csv.gz"))
    .select(
        col("icd_code"),
        col("icd_version").cast("int"),
        col("long_title").alias("icd_desc")
    )
)

print("ICD dictionary rows:", icd_dict_df.count())
icd_dict_df.show(5)

# STEP 4: Filter diagnoses to visits of interest only
# Inner join on hadm_id — keeps only admissions in our timeline
# broadcast(timeline_df) since it is small (1568 rows vs 6M+)
filtered_dx_df = dx_icd_df.join(
    broadcast(timeline_df),
    on="hadm_id",
    how="inner"
)

print("Diagnoses for visits of interest:", filtered_dx_df.count())

# Drop duplicate subject_id introduced by the join
# (both dx_icd_df and timeline_df have subject_id)
filtered_dx_df2 = filtered_dx_df.drop(timeline_df["subject_id"])

# STEP 5: Enrich with ICD descriptions
# Left join on icd_code + icd_version — must match both since
# same code can mean different things in ICD-9 vs ICD-10
diagnoses = (
    filtered_dx_df2.join(
        broadcast(icd_dict_df),  # dictionary is small, broadcast it
        on=["icd_code", "icd_version"],
        how="left"  # keep all rows even if no dictionary entry found
    )
    .select(
        col("subject_id").cast("int"),
        col("hadm_id").cast("int"),
        col("visit_type").cast("string"),
        col("ranking").cast("int"),
        col("icd_code").cast("string"),
        col("icd_version").cast("int"),
        col("icd_desc").cast("string")
    )
    .orderBy("subject_id", "hadm_id", "ranking")
)

print("=== diagnoses DataFrame ===")
print("Row count:", diagnoses.count())
diagnoses.printSchema()
diagnoses.show(10, truncate=False)

# Create Parquet File for later
output_dir = Path("../data/diagnoses")

if output_dir.exists():
    print ("Parquet file exists for diagnoses")
else:
    diagnoses.write.mode("overwrite").parquet(str(output_dir))
    print(f"Wrote results to: {output_dir}")


Timeline visits of interest: 1568
+----------+--------+-----------+
|subject_id| hadm_id| visit_type|
+----------+--------+-----------+
|  10383113|20007405|BC_FIRST_DX|
|  13470381|20010741|BC_FIRST_DX|
|  15129856|20010894|BC_FIRST_DX|
|  13349232|20015647|BC_FIRST_DX|
|  15764116|20020797|    SYMPTOM|
+----------+--------+-----------+
only showing top 5 rows
diagnoses_icd rows: 6364488
+----------+--------+-------+--------+-----------+
|subject_id| hadm_id|ranking|icd_code|icd_version|
+----------+--------+-------+--------+-----------+
|  10000032|22595853|      1|    5723|          9|
|  10000032|22595853|      2|   78959|          9|
|  10000032|22595853|      3|    5715|          9|
|  10000032|22595853|      4|   07070|          9|
|  10000032|22595853|      5|     496|          9|
+----------+--------+-------+--------+-----------+
only showing top 5 rows
ICD dictionary rows: 112107
+--------+-----------+--------------------+
|icd_code|icd_version|            icd_desc|
+--------

In [6]:
# -------------------------------------------------------
# BHOOMIKA'S SECTION for icd_codes lookup table
# Goal: all ICD codes with a status label
# NOT_RELATED = general code
# RELEVANT = symptom related to BC (from symptom_icd_list.txt)
# BC_DIAGNOSIS = confirmed BC code (from bc_icd_codes.csv)
# -------------------------------------------------------

# Step 1: Load full ICD dictionary as base
icd_codes = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(DATA_DIR / "d_icd_diagnoses.csv.gz"))
    .select(
        col("icd_code").cast("string"),
        col("icd_version").cast("int"),
        col("long_title").alias("description")
    )
)

print("Total ICD codes:", icd_codes.count())
icd_codes.show(5)

# Step 2: Load BC diagnosis codes from sprint 2 output
bc_codes_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("../data/bc_icd_codes.csv")
    .select(
        col("icd_code").cast("string"),
        col("icd_version").cast("int")
    )
)

print("BC diagnosis codes:", bc_codes_df.count())
bc_codes_df.show(5)

# Step 3: Load symptom ICD codes from sprint 3 output
symptom_codes_df = (
    spark.read
    .option("header", "false")
    .option("inferSchema", "true")
    .csv("../data/symptom_icd_list.txt")
    .toDF("icd_code", "icd_version")
    .filter(col("icd_code") != "icd_code")
    .select(
        col("icd_code").cast("string"),
        col("icd_version").cast("int")
    )
)

print("Symptom codes:", symptom_codes_df.count())
symptom_codes_df.show(5)

# Step 4: Left join BC codes onto full ICD list
# flag = 1 if it's a BC diagnosis code
icd_with_bc = icd_codes.join(
    broadcast(bc_codes_df.withColumn("is_bc", col("icd_code").isNotNull())),
    on=["icd_code", "icd_version"],
    how="left"
)

# Step 5: Left join symptom codes
# flag = 1 if it's a relevant symptom code
icd_with_both = icd_with_bc.join(
    broadcast(symptom_codes_df.withColumn("is_symptom", col("icd_code").isNotNull())),
    on=["icd_code", "icd_version"],
    how="left"
)

# Step 6: Derive status column
# BC_DIAGNOSIS takes priority over RELEVANT
from pyspark.sql.functions import when

icd_codes = (
    icd_with_both
    .select(
        col("icd_code").cast("string"),
        col("icd_version").cast("int"),
        col("description").cast("string"),
        when(col("is_bc") == True, "BC_DIAGNOSIS")
        .when(col("is_symptom") == True, "RELEVANT")
        .otherwise("NOT_RELATED")
        .alias("status")
    )
    .orderBy("icd_code", "icd_version")
)

print("=== icd_codes lookup table ===")
print("Row count:", icd_codes.count())
icd_codes.printSchema()
icd_codes.show(10, truncate=False)

# Quick check — how many of each status?
print("Status distribution:")
icd_codes.groupBy("status").agg(count("*").alias("count")).show()

output_dir = Path("../data/icd_codes")

if output_dir.exists():
    print ("Parquet file exists for icd_codes")
else:
    icd_codes.write.mode("overwrite").parquet(str(output_dir))
    print(f"Wrote results to: {output_dir}")

Total ICD codes: 112107
+--------+-----------+--------------------+
|icd_code|icd_version|         description|
+--------+-----------+--------------------+
|    0010|          9|Cholera due to vi...|
|    0011|          9|Cholera due to vi...|
|    0019|          9|Cholera, unspecified|
|    0020|          9|       Typhoid fever|
|    0021|          9| Paratyphoid fever A|
+--------+-----------+--------------------+
only showing top 5 rows
BC diagnosis codes: 26
+--------+-----------+
|icd_code|icd_version|
+--------+-----------+
|     C67|         10|
|    C670|         10|
|    C671|         10|
|    C672|         10|
|    C673|         10|
+--------+-----------+
only showing top 5 rows
Symptom codes: 162
+--------+-----------+
|icd_code|icd_version|
+--------+-----------+
|   30653|          9|
|   57400|          9|
|   57401|          9|
|   57410|          9|
|   57411|          9|
+--------+-----------+
only showing top 5 rows
=== icd_codes lookup table ===
Row count: 112107
roo

# -------------------------------------------------------
# Aggregation
# -------------------------------------------------------



In [8]:

# -------------------------------------------------------
# KRISTY'S SECTION
# 1st BC Diagnoses Rankings
# -------------------------------------------------------

# Filter diagnoses to BC_FIRST_DX visits only
bc_dx_visits = diagnoses.filter(col("visit_type") == "BC_FIRST_DX")

# Keep only rows where the icd_code matches a known BC diagnosis code
# Inner join on icd_code + icd_version against bc_codes_df
bc_dx_relevant = bc_dx_visits.join(
    broadcast(bc_codes_df),
    on=["icd_code", "icd_version"],
    how="inner"
)

print("BC diagnosis rows in BC_FIRST_DX visits:", bc_dx_relevant.count())

# Find the highest-ranked (lowest ranking number) BC diagnosis per subject
# ranking 1 = listed first = most important
bc_top_ranking = (
    bc_dx_relevant
    .groupBy("subject_id")
    .agg(spark_min("ranking").alias("top_ranking"))
)

print("Subjects with a BC diagnosis code:", bc_top_ranking.count())

# Aggregate statistics across all subjects' top rankings
bc_box_plot = (
    bc_top_ranking
    .agg(
        mean("top_ranking").alias("MEAN"),
        spark_min("top_ranking").alias("MIN"),
        spark_max("top_ranking").alias("MAX"),
        approx_percentile("top_ranking", 0.25).alias("Q1"),
        approx_percentile("top_ranking", 0.50).alias("Q2 (Median)"),
        approx_percentile("top_ranking", 0.75).alias("Q3"),
        count("top_ranking").alias("Total Count")
    )
)

print("BC Diagnosis Box Plot")
bc_box_plot.show(truncate=False)


BC diagnosis rows in BC_FIRST_DX visits: 1171
Subjects with a BC diagnosis code: 1143
BC Diagnosis Box Plot
+-----------------+---+---+---+-----------+---+-----------+
|MEAN             |MIN|MAX|Q1 |Q2 (Median)|Q3 |Total Count|
+-----------------+---+---+---+-----------+---+-----------+
|5.619422572178478|1  |35 |1  |4          |8  |1143       |
+-----------------+---+---+---+-----------+---+-----------+



In [9]:

# -------------------------------------------------------
# BHOOMIKA'S SECTION — Symptom Diagnosis Rankings
# Goal: Check how relevant symptom visits are by finding
# the minimum ranking of relevant ICD codes per visit
# -------------------------------------------------------

# Filter diagnoses to symptom visits only
symptom_visits_df = diagnoses.filter(col("visit_type") == "SYMPTOM")

# Get only RELEVANT codes from our icd_codes lookup table
relevant_icd_df = icd_codes.filter(col("status") == "RELEVANT")

# Inner join — keep only symptom rows where icd_code is RELEVANT
relevant_symptoms_df = symptom_visits_df.join(
    broadcast(relevant_icd_df),
    on=["icd_code", "icd_version"],
    how="inner"
)

print("Relevant symptom rows:", relevant_symptoms_df.count())
relevant_symptoms_df.show(5)

# Group by visit (hadm_id) and find minimum ranking per visit
# Lower ranking = more important diagnosis
symptom_rankings = (
    relevant_symptoms_df
    .groupBy("hadm_id")
    .agg(
        spark_min(col("ranking")).alias("symptom_rankings")
    )
)

print("=== Symptom Rankings Summary ===")
symptom_rankings.select("symptom_rankings").summary().show()


Relevant symptom rows: 547
+--------+-----------+----------+--------+----------+-------+--------------------+--------------------+--------+
|icd_code|icd_version|subject_id| hadm_id|visit_type|ranking|            icd_desc|         description|  status|
+--------+-----------+----------+--------+----------+-------+--------------------+--------------------+--------+
|   R1031|         10|  10120826|27121829|   SYMPTOM|      4|Right lower quadr...|Right lower quadr...|RELEVANT|
|    5990|          9|  10247438|23745352|   SYMPTOM|      2|Urinary tract inf...|Urinary tract inf...|RELEVANT|
|   78820|          9|  10247438|23745352|   SYMPTOM|     12|Retention of urin...|Retention of urin...|RELEVANT|
|    R310|         10|  10247438|29483315|   SYMPTOM|     14|     Gross hematuria|     Gross hematuria|RELEVANT|
|    5990|          9|  10255052|23614192|   SYMPTOM|      2|Urinary tract inf...|Urinary tract inf...|RELEVANT|
+--------+-----------+----------+--------+----------+-------+--------

In [10]:
# -------------------------------------------------------
# AGE and DATE Transformations (Ara)
# -------------------------------------------------------

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# subtask 1: Frequency of Age at Diagnosis

bc_visits = visits.filter(F.col("visit_type") == "BC_FIRST_DX")

bc_buckets = bc_visits.withColumn(
    "de_obfs_age_bucket",
    F.when(F.col("age") < 30, "<30")
     .when((F.col("age") >= 30) & (F.col("age") <= 40), "30-40")
     .when((F.col("age") >= 41) & (F.col("age") <= 55), "41-55")
     .when((F.col("age") >= 56) & (F.col("age") <= 70), "56-70")
     .when((F.col("age") >= 71) & (F.col("age") <= 85), "71-85")
     .otherwise("85+")
)

age_frequency = (
    bc_buckets
    .groupBy("de_obfs_age_bucket")
    .count()
    .orderBy("de_obfs_age_bucket")
)

age_frequency.show(truncate=False)


# subtask 2: Days Prior to Diagnosis

# using the timeline here because it preserves symptom/diagnosis event order
symptoms = pre_bc_symptom_timeline.filter(F.col("row_type") == "SYMPTOM")
bc = pre_bc_symptom_timeline.filter(F.col("row_type") == "BC_FIRST_DX")

symptom_counts = (
    symptoms
    .groupBy("subject_id")
    .count()
    .withColumnRenamed("count", "num_of_symptoms")
)

the_window = Window.partitionBy("subject_id").orderBy(F.col("admittime").desc())

last_symptoms = (
    symptoms
    .withColumn("rank", F.row_number().over(the_window))
    .filter(F.col("rank") == 1)
    .select("subject_id", F.col("admittime").alias("last_symptom_time"))
)

first_bc = (
    bc
    .select("subject_id", F.col("admittime").alias("bc_time"))
)

time_difference = (
    last_symptoms
    .join(first_bc, on="subject_id", how="inner")
    .join(symptom_counts, on="subject_id", how="inner")
    .withColumn(
        "days_before_dx",
        F.datediff("bc_time", "last_symptom_time")
    )
)

time_difference = time_difference.withColumn(
    "bucket",
    F.when(F.col("num_of_symptoms") == 1, "1")
     .when(F.col("num_of_symptoms") == 2, "2")
     .otherwise("3+")
)

stats_bucket = (
    time_difference
    .groupBy("bucket")
    .agg(
        F.mean("days_before_dx").alias("mean"),
        F.stddev("days_before_dx").alias("stddev"),
        F.max("days_before_dx").alias("max"),
        F.min("days_before_dx").alias("min"),
        F.expr("percentile_approx(days_before_dx, 0.5)").alias("median")
    )
    .orderBy("bucket")
)

stats_bucket.show(truncate=False)

+------------------+-----+
|de_obfs_age_bucket|count|
+------------------+-----+
|30-40             |8    |
|41-55             |67   |
|56-70             |320  |
|71-85             |530  |
|85+               |218  |
+------------------+-----+

+------+-----------------+------------------+----+---+------+
|bucket|mean             |stddev            |max |min|median|
+------+-----------------+------------------+----+---+------+
|1     |747.776          |1047.7206047169193|5104|3  |252   |
|2     |405.9130434782609|580.9278263142412 |3325|6  |116   |
|3+    |321.344262295082 |525.3552095248161 |2206|2  |84    |
+------+-----------------+------------------+----+---+------+



In [11]:
# -------------------------------------------------------
# AGE and DATE Transformations (Ara)
# -------------------------------------------------------

# My section performs two analyses:
# 1) Counting how many BC diagnosis visits fall into each age range -> named Subtask 1 
# 2) Measuring how many days passed between a patient's last symptom visit
#    and their bladder cancer diagnosis visit -> named Subtask 2 

# SUBTASK 1: Frequency of Age at Diagnosis



# Keeping only visits labeled as the first breast cancer diagnosis
# Each row in 'visits' represents a visit of interest with demographic info attached
bc_visits = visits.filter(F.col("visit_type") == "BC_FIRST_DIAGNOSIS")

# Create age buckets from the derived age column
# The 'age' field was computed earlier from anchor_age and anchor_year,
# so this is a grouped version of that approximate/de-obfuscated age
bc_buckets = bc_visits.withColumn(
    "age_bucket",
    F.when(F.col("age") < 30, "<30")
     .when((F.col("age") >= 30) & (F.col("age") <= 40), "30-40")
     .when((F.col("age") >= 41) & (F.col("age") <= 55), "41-55")
     .when((F.col("age") >= 56) & (F.col("age") <= 70), "56-70")
     .when((F.col("age") >= 71) & (F.col("age") <= 85), "71-85")
     .otherwise("85+")
)

# Count how many BC diagnosis visits fall into each age bucket
age_frequency = (
    bc_buckets
    .groupBy("age_bucket")
    .count()
    .orderBy("age_bucket")
)

# Display the age distribution table
age_frequency.show(truncate=False)


# SUBTASK 2: Days Prior to Diagnosis


# Use the timeline dataframe because it preserves the ordering of
# symptom visits and diagnosis visits for each patient

# Keep only symptom rows
symptoms = pre_bc_symptom_timeline.filter(F.col("row_type") == "SYMPTOM")

# Keeping only breast cancer diagnosis rows
bc = pre_bc_symptom_timeline.filter(F.col("row_type") == "BC_FIRST_DX")

# Counting how many symptom visits each patient has in the timeline
# This will later let us compare patients with 1, 2, or 3+ symptoms
symptom_counts = (
    symptoms
    .groupBy("subject_id")
    .count()
    .withColumnRenamed("count", "num_of_symptoms")
)

# `window` definition
# Paritioning by patient and sort each patient's symptom visits by admittime descending
# Which allows us to identify the most recent symptom visit for each patient
from pyspark.sql.window import Window
the_window = Window.partitionBy("subject_id").orderBy(F.col("admittime").desc())

# For each patient, keep only the most recent symptom visit
# row_number() = 1 means "latest symptom visit" because of descending sort
last_symptoms = (
    symptoms
    .withColumn("rank", F.row_number().over(the_window))
    .filter(F.col("rank") == 1)
    .select("subject_id", F.col("admittime").alias("last_symptom_time"))
)

# Get the breast cancer diagnosis time for each patient
# Assumes one BC_FIRST_DX row per patient in this timeline
first_bc = (
    bc
    .select("subject_id", F.col("admittime").alias("bc_time"))
)

# Join:
# - the patient's last symptom time
# - the patient's diagnosis time
# - the number of symptoms that patient had
# Then compute the number of days between the last symptom visit
# and the breast cancer diagnosis visit
time_difference = (
    last_symptoms
    .join(first_bc, on="subject_id", how="inner")
    .join(symptom_counts, on="subject_id", how="inner")
    .withColumn(
        "days_before_dx",
        F.datediff("bc_time", "last_symptom_time")
    )
)

# Bucketing patients by number of symptoms:
# 1 symptom, 2 symptoms, or 3+ symptoms
time_difference = time_difference.withColumn(
    "bucket",
    F.when(F.col("num_of_symptoms") == 1, "1")
     .when(F.col("num_of_symptoms") == 2, "2")
     .otherwise("3+")
)

# For each symptom-count bucket, summary statistics are computed for days_before_dx:
# - mean
# - standard deviation
# - maximum
# - minimum
# - approximate median

# Note: the following stats were specified accordingly in the Trello Card 
stats_bucket = (
    time_difference
    .groupBy("bucket")
    .agg(
        F.mean("days_before_dx").alias("mean"),
        F.stddev("days_before_dx").alias("stddev"),
        F.max("days_before_dx").alias("max"),
        F.min("days_before_dx").alias("min"),
        F.expr("percentile_approx(days_before_dx, 0.5)").alias("median")
    )
    .orderBy("bucket")
)

# Displaying the stats for each bucket 
stats_bucket.show(truncate=False)

+----------+-----+
|age_bucket|count|
+----------+-----+
+----------+-----+

+------+-----------------+------------------+----+---+------+
|bucket|mean             |stddev            |max |min|median|
+------+-----------------+------------------+----+---+------+
|1     |747.776          |1047.7206047169193|5104|3  |252   |
|2     |405.9130434782609|580.9278263142412 |3325|6  |116   |
|3+    |321.344262295082 |525.3552095248161 |2206|2  |84    |
+------+-----------------+------------------+----+---+------+



## Clean up
Stop spark session when done

In [13]:
# Uncomment when you are completely done:

spark.stop()